# A8 đầy đủ: train bản nộp với che tên app và tool, gộp LoRA, đo dev (khoảng **4 đến 4,5 giờ** trên RTX PRO 6000)
Đây là bản **thử cuối** cho bài nộp (Phương quyết định ngày 15/11). Công thức: `q4b_lora` (sft_train + sft_train_asr, 3 epoch) **cộng** 100 mẫu oos (`sft_train_aug`) và 1390 bản sao che tên (`sft_train_mask`), train trên **toàn bộ train** (không giấu app nào), giữ nguyên bộ chia dev 155 mẫu. Rồi gộp LoRA và đo dev (chữ sạch và ASR `phowl_ft`) để so với mốc **66,82 và 56,98**. Hướng dẫn đầy đủ, mốc giờ và việc sau khi train: `docs/HUONG_DAN_A8_FULL.md`.

**Chỉ chạy trên GPU RTX PRO 6000 (1 GPU).** Trên L4 x4 train mất 7 đến 8 giờ, không kịp hạn: cell F-3 dừng nếu không phải 1 GPU (đặt `ALLOW_SLOW=1` chỉ khi Phương đồng ý).
**Trước khi chạy:** notebook **Private**, Internet **Off**, Accelerator **GPU RTX PRO 6000**.
**Input cần gắn (4 thứ):**
1. dataset `lifemate-code` bản **có `src/lifemate/masking.py`** (cùng bản đã dùng cho A8, commit `d5f376c` trở lên)
2. dataset `train+public_test` (raw)
3. dataset `thuan-qwen3-4b`
4. **output của notebook `Hello_done`** (adapter `q4b_lora`, `state/processed` có `asr_phowl_train.jsonl` và `asr_phowl_ft_train.jsonl`)

**KHÔNG gắn** output A4, A4b, A5, A8 hay `merge-lora` (cùng có `state/`, gây khôi phục nhầm).
**Thứ tự cell:** P-1 đến P-3, F-1 (kiểm tra), F-2 (dựng dữ liệu, preflight), F-3 (train), F-4 (gộp), F-5 (đo dev), F-6 (phán quyết), F-7 (zip), F-8 (dọn, chạy cuối rồi Save Version).
**Khi xong:** gửi Phương `a8_full_results.zip` và dòng phán quyết ở F-6; `checkpoints/q4b_a8_merged/*` (khoảng 8 GB) **chỉ chuyển đi khi phán quyết ĐẠT**. Kèm sha256 của `model.safetensors`. Không xóa notebook và output cho đến hết cuộc thi.
**Nếu train lỗi hết bộ nhớ hoặc dừng sớm** (chạm `--max-hours 5` trước khi đủ 3 epoch): gửi nguyên văn 40 dòng cuối log, **không** tự đổi tham số; bản dừng sớm không dùng được.

In [ ]:
%%bash
# P-1: setup. Dùng kaggle_setup.sh của Phương; chỉ đặt đường dẫn model cho khớp đúng thứ ta gắn:
#  - QWEN = Qwen3-4B (hidden 2560, 36 tầng), tìm theo cấu hình vì còn có Qwen3-8B/Qwen3.5-9B trong input (không lấy "qwen3 đầu tiên")
#  - PHO = đường dẫn giả (A5 không dùng PhoWhisper)
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
find /kaggle/input -maxdepth 8 -name config.json 2>/dev/null > /kaggle/working/config_paths.txt
QW=$(python -X utf8 - <<'EOF'
import json
from pathlib import Path
for line in open("/kaggle/working/config_paths.txt", encoding="utf-8"):
    p = Path(line.strip())
    try:
        c = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue
    t = c.get("text_config", c)
    if c.get("model_type") == "qwen3" and t.get("hidden_size") == 2560 and t.get("num_hidden_layers") == 36:
        print(p.parent)
        break
EOF
)
echo "Qwen3-4B dir: ${QW:-KHÔNG THẤY}"
[ -n "$QW" ] || { echo "ERROR: không thấy Qwen3-4B (hidden 2560, 36 tầng). Gắn dataset thuan-qwen3-4b."; exit 1; }
cp "$SRC" /kaggle/working/setup_patched.sh
sed -i -e "s#^QWEN=.*#QWEN=$QW#" -e 's#^PHO=.*#PHO=/nonexistent/phowhisper#' -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "--- đã khôi phục từ output notebook trước? (checkpoints và outputs)"; ls /kaggle/working/checkpoints | head -20; ls /kaggle/working/outputs | head -12

In [ ]:
%%bash
# P-2: in phiên bản torch/CUDA/GPU, kiểm tra GPU, peft, accelerate và việc transformers nhận qwen3_5
pip uninstall -y torchao 2>&1 | tail -1
python -X utf8 - <<'EOF' | tee /kaggle/working/torch_cuda.txt
import sys
import torch
print(torch.__version__, torch.version.cuda, sys.version)
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__, "| gpus", torch.cuda.device_count(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("transformers nhận qwen3:", "qwen3" in CONFIG_MAPPING, "| qwen3_5 (Qwen3.5):", "qwen3_5" in CONFIG_MAPPING)
try:
    import accelerate
    print("accelerate", accelerate.__version__)
except ImportError:
    print("THIẾU accelerate: gen_text.py sẽ lỗi ở bước nạp model (device_map). Báo lại, đừng pip install từ mạng.")
if not torch.cuda.is_available():
    raise SystemExit("Phiên này KHÔNG có GPU: bật Accelerator rồi chạy lại")
print("VRAM tổng: %.0f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
EOF

In [ ]:
%%bash
# P-3: ghi sẵn đoạn mã đóng gói (chỉ file kết quả nhỏ; không có trọng số)
mkdir -p /kaggle/working/helpers
cat > /kaggle/working/helpers/pack_zip.py <<'PYEOF'
"""Đóng gói file kết quả nhỏ thành một zip kèm MANIFEST sha256 (không đóng gói trọng số).
usage: python pack_zip.py <out.zip> <file> [<file> ...]   (file không tồn tại thì bỏ qua và báo)"""
import hashlib
import sys
import zipfile
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()


out = Path(sys.argv[1])
files = []
for name in sys.argv[2:]:
    p = Path(name)
    if p.is_file():
        files.append(p)
    else:
        print("bỏ qua (không có):", name)
files = sorted(set(files), key=str)
manifest = "".join(f"{sha256(p)}  {p.as_posix()}  ({p.stat().st_size} bytes)\n" for p in files)
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.as_posix().lstrip("/"))
    z.writestr("MANIFEST_sha256.txt", manifest)
print(f"{out}: {out.stat().st_size / 1e6:.2f} MB, {len(files)} file")
PYEOF
ls -la /kaggle/working/helpers

In [ ]:
%%bash
# F-1: kiểm tra đủ đồ cho A8 ĐẦY ĐỦ: mã mới (masking.py, --aug-mask), ASR phowl và phowl_ft của train, adapter q4b_lora làm mốc, dev_ids. Thiếu thì DỪNG.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
MISSING=0
need() { if [ -s "$1" ]; then echo "ok     $1"; else echo "THIẾU  $1"; MISSING=1; fi; }
need scripts/build_sft.py
need scripts/train_sft.py
need scripts/preflight_train.py
need scripts/merge_lora.py
need scripts/infer_multi_gpu.sh
need src/lifemate/masking.py
need data/processed/asr_phowl_train.jsonl
need data/processed/asr_phowl_ft_train.jsonl
need data/processed/dev_ids.txt
need data/processed/gold_hash_train.json
need checkpoints/q4b_lora/adapter_model.safetensors
if grep -q -- "--aug-mask" scripts/build_sft.py 2>/dev/null; then echo "ok     build_sft.py có --aug-mask"; else echo "THIẾU  build_sft.py chưa có --aug-mask: dùng lifemate-code mới"; MISSING=1; fi
if [ "$MISSING" = 1 ]; then echo "-> Kiểm tra input: lifemate-code mới, train+public_test, thuan-qwen3-4b, output notebook Hello_done."; exit 1; fi
n=$(wc -l < data/processed/dev_ids.txt)
[ "$n" = 155 ] || { echo "dev_ids.txt có $n dòng, phải là 155"; exit 1; }
echo "dev_ids.txt: 155 id (đúng)"
cp data/processed/dev_ids.txt /kaggle/working/dev_ids_before.txt        # F-2 sẽ so lại: bộ chia dev phải KHÔNG đổi
echo "-- test của phần che tên (nhanh, không cần GPU)"
python -X utf8 -m pytest -q tests/test_masking.py 2>&1 | tail -3
echo "-- dọn bản sao không dùng (giữ q4b_lora làm mốc)"
for d in checkpoints/*; do case "$(basename "$d")" in q4b_lora) ;; *) rm -rf "$d" ;; esac; done
rm -rf checkpoints/q4b_lora/checkpoint-* /kaggle/working/outputs/*
mkdir -p outputs/logs
df -h /kaggle/working | tail -1

In [ ]:
%%bash
# F-2: dựng dữ liệu TOÀN BỘ train: sft_train, sft_train_asr (như q4b_lora) + sft_train_aug (100 oos) + sft_train_mask (1390 bản che tên).
# Kỳ vọng: 1390 train, 155 dev, "augmentation -> sft_train_aug.jsonl: 100 samples", "name masking -> sft_train_mask.jsonl: 1390 samples". Bộ chia dev PHẢI trùng bản cũ.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
python -X utf8 scripts/build_sft.py --asr-tag phowl --aug-oos 100 --aug-mask 1390 2>&1 | cut -c1-400 | tee outputs/logs/a8f_build.txt
D=data/processed
cmp $D/dev_ids.txt /kaggle/working/dev_ids_before.txt || { echo "BỘ CHIA DEV ĐÃ ĐỔI: dừng, gửi output này cho Phương (không train)"; exit 1; }
echo "dev_ids.txt trùng bản trước khi dựng (155 id)"
wc -l $D/sft_train.jsonl $D/sft_train_asr.jsonl $D/sft_train_aug.jsonl $D/sft_train_mask.jsonl $D/dev_ids.txt
[ "$(wc -l < $D/sft_train.jsonl)" = 1390 ] || { echo "sft_train.jsonl không đủ 1390 mẫu: dừng"; exit 1; }
[ "$(wc -l < $D/sft_train_aug.jsonl)" = 100 ] || { echo "sft_train_aug.jsonl không đủ 100 mẫu: dừng"; exit 1; }
[ "$(wc -l < $D/sft_train_mask.jsonl)" = 1390 ] || { echo "sft_train_mask.jsonl không đủ 1390 mẫu: dừng"; exit 1; }
sha256sum $D/sft_train.jsonl $D/sft_train_asr.jsonl $D/sft_train_aug.jsonl $D/sft_train_mask.jsonl | tee -a outputs/logs/a8f_build.txt
python -X utf8 scripts/preflight_train.py --data $D/sft_train.jsonl $D/sft_train_asr.jsonl $D/sft_train_aug.jsonl $D/sft_train_mask.jsonl \
  --dev-ids $D/dev_ids.txt --max-len 5120 --tokenizer $QWEN 2>&1 | tee outputs/logs/a8f_preflight.txt

In [ ]:
%%bash
# F-3: TRAIN q4b_a8_full = sft_train + sft_train_asr + sft_train_aug + sft_train_mask, 3 epoch, batch tổng 16, max-len 5120, như q4b_lora (+ hai nguồn dữ liệu thêm).
# Khoảng 3,5 đến 4 giờ trên RTX PRO 6000. --max-hours 5: nếu chạm giới hạn thì CHƯA đủ 3 epoch và bản này BỎ (cosine LR chưa hạ hết).
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
D=data/processed
NGPU=$(python -c "import torch; print(torch.cuda.device_count())")
if [ "$NGPU" != 1 ] && [ "${ALLOW_SLOW:-0}" != 1 ]; then
  echo "Có $NGPU GPU: train toàn bộ trên L4 x4 mất 7 đến 8 giờ, không kịp hạn Test Solution. DỪNG (ALLOW_SLOW=1 chỉ khi Phương đồng ý)."; exit 1
fi
nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader | tee outputs/logs/a8f_gpus.txt
START=$(date +%s)
torchrun --nproc_per_node=$NGPU scripts/train_sft.py --model $QWEN \
  --data $D/sft_train.jsonl $D/sft_train_asr.jsonl $D/sft_train_aug.jsonl $D/sft_train_mask.jsonl \
  --out checkpoints/q4b_a8_full --epochs 3 --max-len 5120 --global-batch 16 --save-limit 2 --max-hours 5 \
  > outputs/logs/a8f_train.log 2>&1 || { echo "TRAIN LỖI (nguyên văn cuối log):"; tail -n 40 outputs/logs/a8f_train.log; exit 1; }
grep -v "it/s\]" outputs/logs/a8f_train.log | tail -n 8 || true
ELAPSED=$(( $(date +%s) - START ))
echo "thời gian train: $ELAPSED s trên $NGPU GPU" | tee outputs/logs/a8f_train_time.txt
if [ "$ELAPSED" -ge 17800 ]; then
  echo "CHẠM GIỚI HẠN --max-hours 5: CHƯA ĐỦ 3 EPOCH, BẢN NÀY BỎ. Báo Phương." | tee -a outputs/logs/a8f_train_time.txt
fi
ls checkpoints/q4b_a8_full
sha256sum checkpoints/q4b_a8_full/adapter_model.safetensors | tee outputs/logs/a8f_adapter_sha256.txt

In [ ]:
%%bash
# F-4 (khoảng 5 đến 10 phút, cần khoảng 20 GB RAM): GỘP LoRA vào Qwen3-4B -> checkpoints/q4b_a8_merged (gộp float32, lưu bfloat16), rồi sha256.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
START=$(date +%s)
python -X utf8 scripts/merge_lora.py --base "$QWEN" --adapter checkpoints/q4b_a8_full --out checkpoints/q4b_a8_merged 2>&1 | tee outputs/logs/a8f_merge.txt | tail -n 5
echo "thời gian gộp: $(( $(date +%s) - START )) s" | tee -a outputs/logs/a8f_merge.txt
ls -la checkpoints/q4b_a8_merged
( cd checkpoints && find q4b_a8_merged -type f | sort | xargs sha256sum ) | tee /kaggle/working/SHA256SUMS_a8_merged.txt
du -sh checkpoints/q4b_a8_merged

In [ ]:
%%bash
# F-5 (khoảng 12 phút): đo dev (155 mẫu) với model ĐÃ GỘP, không adapter: chữ sạch và ASR phowl_ft, luật bật, đúng lệnh của merge-lora.
# Mốc cần so: 66,82 (sạch) và 56,98 (ASR phowl_ft) của q4b_lora đã gộp.
set -euo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
D=data/processed
bash scripts/infer_multi_gpu.sh train outputs/dev_a8_merged_clean.jsonl --ids $D/dev_ids.txt --backend hf --model checkpoints/q4b_a8_merged --text clean
python -X utf8 scripts/eval.py --sub outputs/dev_a8_merged_clean.jsonl --ids $D/dev_ids.txt | tee outputs/logs/a8f_eval_clean.txt
bash scripts/infer_multi_gpu.sh train outputs/dev_a8_merged_phowl_ft.jsonl --ids $D/dev_ids.txt --backend hf --model checkpoints/q4b_a8_merged --text asr:phowl_ft
python -X utf8 scripts/eval.py --sub outputs/dev_a8_merged_phowl_ft.jsonl --ids $D/dev_ids.txt | tee outputs/logs/a8f_eval_phowl_ft.txt
tail -qn 1 outputs/dev_a8_merged_clean.part*.log outputs/dev_a8_merged_phowl_ft.part*.log | tee outputs/logs/a8f_timing.txt

In [ ]:
%%bash
# F-6: PHÁN QUYẾT (chỉ đọc số tổng hợp). ĐẠT nếu: (1) dev sạch >= 66,82; (2) dev ASR phowl_ft >= 56,98; (3) ask (sạch) >= 70,64 (mốc 75,64 trừ 5: A5 và A8 đều làm ask tụt).
# ĐẠT chỉ nghĩa là ĐỦ ĐIỀU KIỆN để cân nhắc; Phương quyết định có nộp hay không (dev 155 mẫu nhiễu, +-4 điểm).
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
python -X utf8 - <<'EOF' | tee outputs/logs/a8f_verdict.txt
import re
def read(p):
    t = open(p, encoding="utf-8").read()
    total = float(re.search(r"score=\s*([0-9.]+)", t).group(1))
    kinds = {k: float(v) for k, v in re.findall(r"kind=(\w+)\s+n=\s*\d+\s+mean=\s*([0-9.]+)", t)}
    return total, kinds
c, ck = read("outputs/logs/a8f_eval_clean.txt")
a, ak = read("outputs/logs/a8f_eval_phowl_ft.txt")
print(f"dev sạch      : {c:6.2f}  (mốc 66,82)  ask {ck.get('ask', float('nan')):6.2f} (mốc 75,64)  exec {ck.get('exec', float('nan')):6.2f} (65,73)  oos {ck.get('oos', float('nan')):6.2f} (58,33)")
print(f"dev ASR ft    : {a:6.2f}  (mốc 56,98)  ask {ak.get('ask', float('nan')):6.2f}  exec {ak.get('exec', float('nan')):6.2f}  oos {ak.get('oos', float('nan')):6.2f}")
ok1, ok2, ok3 = c >= 66.82, a >= 56.98, ck.get("ask", 0) >= 70.64
print(f"(1) sạch >= 66,82: {'ĐẠT' if ok1 else 'KHÔNG'}   (2) ASR >= 56,98: {'ĐẠT' if ok2 else 'KHÔNG'}   (3) ask >= 70,64: {'ĐẠT' if ok3 else 'KHÔNG'}")
print("=> " + ("A8 ĐẦY ĐỦ ĐẠT điều kiện dev: gửi sha256 và chuyển q4b_a8_merged cho người dựng image (Phương quyết)" if ok1 and ok2 and ok3 else "A8 ĐẦY ĐỦ KHÔNG đạt: GIỮ bài nộp hiện tại, không chuyển trọng số"))
EOF

In [ ]:
%%bash
# F-7: gói kết quả nhỏ để gửi Phương: a8_full_results.zip (điểm, phán quyết, log, sha256 của model gộp, môi trường). KHÔNG có trọng số.
set -uo pipefail
source /kaggle/working/lifemate/kaggle_env.sh
python -X utf8 /kaggle/working/helpers/pack_zip.py /kaggle/working/a8_full_results.zip \
  outputs/dev_a8_merged_clean.jsonl outputs/dev_a8_merged_phowl_ft.jsonl outputs/logs/a8f_*.txt outputs/logs/a8f_*.log \
  /kaggle/working/SHA256SUMS_a8_merged.txt /kaggle/working/torch_cuda.txt
ls -la /kaggle/working/a8_full_results.zip

In [ ]:
%%bash
# F-8 (CUỐI CÙNG): dọn rồi Save Version. Giữ checkpoints/q4b_a8_merged (khoảng 8 GB), adapter q4b_a8_full (nhỏ), outputs/, a8_full_results.zip, SHA256SUMS_a8_merged.txt.
# KHÔNG chạy kaggle_save_state.sh. Không xóa notebook và output cho đến hết cuộc thi.
cd /kaggle/working
rm -rf checkpoints/q4b_a8_full/checkpoint-* checkpoints/q4b_lora
rm -rf lifemate helpers setup_patched.sh config_paths.txt state dev_ids_before.txt
du -sh /kaggle/working/* 2>/dev/null
ls /kaggle/working /kaggle/working/checkpoints
echo "xong. Save Version (private). Tab Output: a8_full_results.zip (gửi Phương), SHA256SUMS_a8_merged.txt và checkpoints/q4b_a8_merged/* (chỉ chuyển nếu phán quyết ĐẠT)."